# AI-Powered Resource Recommendation System
## Part 1: Data Discovery, Cleaning, & Validation Pipeline

This notebook serves as the master documentation for the data pipeline. It explains the steps taken to profile the raw hackathon datasets, validate relationships, clean data anomalies, and prepare clean outputs for downstream machine learning and recommendation models.

### Project Directory Structure
```
project/
├── rawData/                  (Read-only raw data files)
├── cleanedData/              (Deduplicated, parsed, standardized outputs)
├── cleaning/                 (Modular python scripts)
│   ├── config.py             (Paths and mappings)
│   ├── utils.py              (Date parsing, experience converter, type cast helpers)
│   ├── clean_data.py         (Main orchestrator script)
│   ├── validation.py         (Post-cleaning validation checks)
│   └── feature_engineering.py(Feature recommendations)
└── notebooks/
    └── 01_data_profiling.ipynb (This documentation)
```

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from pathlib import Path

clean_dir = Path("../cleaning")
cleaned_data_dir = Path("../cleanedData")
reports_dir = Path("../cleaning/reports")

### 1. Dataset Profiling Summary
We profile raw file structures (row count, columns, duplicates, memory footprints, primary and foreign key candidates) to check database shapes before cleaning.

In [ ]:
df_summary = pd.read_csv(clean_dir / "dataset_summary.csv")
df_summary

### 2. Data Dictionary
A detailed column-level breakdown showing unique values, null count, null percentage, data ranges, and sample values.

In [ ]:
df_dict = pd.read_csv(clean_dir / "data_dictionary.csv")
display(df_dict.head(10))
print(f"Total columns defined in schema: {len(df_dict)}")

### 3. Missing Value Analysis
We identify missing values by column to decide on appropriate cleaning and replacement strategies.

In [ ]:
display(Image(filename=reports_dir / "missing_values.png"))
display(Image(filename=reports_dir / "null_percentage.png"))

### 4. Duplicate Analysis
We look for duplicate rows and keys to prevent data leakage in training datasets.

In [ ]:
df_dups = pd.read_csv(clean_dir / "duplicate_report.csv")
display(df_dups)
display(Image(filename=reports_dir / "duplicate_summary.png"))

### 5. Relationship Analysis & Referential Integrity
We evaluate primary/foreign key mappings (e.g. employee_id references from allocations to employee master) and identify broken references.

In [ ]:
df_rel = pd.read_csv(clean_dir / "relationship_report.csv")
df_rel

### 6. Cleaning Strategies Implemented

The following operations were implemented programmatically in `cleaning/clean_data.py`:

1. **Column Standardizations**: Convert columns to lowercase snake_case (e.g., `Employee ID` $\rightarrow$ `employee_id`).
2. **Text Normalization**: Clean spacing, remove trailing characters, and map variations (e.g., `Python`/`PYTHON` $\rightarrow$ `Python`, `ReactJS`/`React.js` $\rightarrow$ `React`). ReAct AI prompting framework was protected from frontend React mapping.
3. **Date Standardizations**: Parsed DD-MM-YYYY strings and formatted consistently to standard `YYYY-MM-DD` datetimes.
4. **Casting Experience ranges**: Strings like `'1-2 Year'` were parsed into numeric midpoint averages (`1.5`) for modeling compatibility.
5. **Casting Allocation percentages**: Strings like `'75%'` were parsed into numeric integers (`75`).
6. **Consolidation of Competencies**: The three Excel sheets representing Solution Enablers, Consultants, and Senior Engineers were merged into a single standardized table (`competencies_clean.csv`) aligning overlapping scoring dimensions.
7. **Consolidation of Pipeline**: Extracted and cleaned the `Forecast` sheet into `pipeline_clean.csv`.
8. **Handling Impossible Values**: Six allocations where the end date occurred before the start date were flagged using `impossible_value_flag = 1` rather than silently dropped.
9. **Null Value Handling**: Missing locations, job designations, and managers were mapped to `'Unknown'`. Active employees' resignation dates were left as null/NaT.

### 7. Data Quality Scores (out of 100)
Quality Scores are computed as the average of Completeness, Consistency (no impossible values), Uniqueness (no duplicate primary keys), and Integrity (no orphaned references).

In [ ]:
df_quality = pd.read_csv(clean_dir / "quality_report.csv")
df_quality

### 8. Programmatic Validation Assertions
We run assertion checks using `cleaning/validation.py`. The output verification status log is displayed below:

In [ ]:
with open(clean_dir / "validation_report.txt", "r", encoding="utf-8") as f:
    print(f.read())

### 9. Clean Data Exploratory Visuals

In [ ]:
display(Image(filename=reports_dir / "dataset_sizes.png"))
display(Image(filename=reports_dir / "top_skills.png"))
display(Image(filename=reports_dir / "top_competencies.png"))
display(Image(filename=reports_dir / "allocation_distribution.png"))

### 10. Downstream Feature Recommendations
We recommend engineered features for resources and projects, exported as `feature_recommendations.csv`:

In [ ]:
df_feats = pd.read_csv(clean_dir / "feature_recommendations.csv")
df_feats